# Scenario: Tracing the Shifting Ground Truth

In [1]:
import pandas as pd
import sqlite3
# creating dataset capturing when a clinical reading occurred vs when it was actually written to the DB
record_drift_data = {
    "log_id": [8001, 8002, 8003, 8004, 8005],
    "patient_id": ["P-61", "P-62", "P-63", "P-64", "P-65"],
    "clinical_event_date": ["2026-05-10", "2026-05-11", "2026-05-12", "2026-05-15", "2026-05-16"],
    # This column tracks the actual date the row was modified/inserted into the database
    "db_system_write_date": ["2026-05-10", "2026-06-02", "2026-05-12", "2026-05-15", "2026-06-04"]
    # Look closely at 8002 and 8005! They were written weeks after the patient visit occurred!
}
# adding dataset to DataFrame
df_record_drift_data = pd.DataFrame(record_drift_data)
# creating sql and saving dataframe to temp memory
connt = sqlite3.connect(":memory:")
df_record_drift_data.to_sql("clinical_snapshots", connt, index = False, if_exists = "replace")
# function to run the query
def run_query(query):
    return pd.read_sql_query(query, connt)
print("******************************** Record Drift Audit Database is ready! **************")

******************************** Record Drift Audit Database is ready! **************


# Detecting Retrospective Back-Dating

In [4]:
# all data to review
all_data = "SELECT * FROM clinical_snapshots"
print("******************************* all data to review ************")
display(run_query(all_data))
print()
# SQL query that uses date comparison to locate any records where the db_system_write_date is strictly greater than the clinical_event_date.
date_comparison = """
SELECT 
    patient_id,
    clinical_event_date,
    db_system_write_date   
FROM clinical_snapshots
WHERE db_system_write_date > clinical_event_date
    
"""
print("************************* date_comparison between clinical_event_date and db_system_write_date ******************")
display(run_query(date_comparison))

******************************* all data to review ************


,log_id,patient_id,clinical_event_date,db_system_write_date
0,8001,P-61,2026-05-10,2026-05-10
1,8002,P-62,2026-05-11,2026-06-02
2,8003,P-63,2026-05-12,2026-05-12
3,8004,P-64,2026-05-15,2026-05-15
4,8005,P-65,2026-05-16,2026-06-04



************************* date_comparison between clinical_event_date and db_system_write_date ******************


,patient_id,clinical_event_date,db_system_write_date
0,P-62,2026-05-11,2026-06-02
1,P-65,2026-05-16,2026-06-04


# Quantifying the Modification Delay (in Days)

In [5]:
# SQL query to isolate these delayed entries and calculate the exact number of days that elapsed between the patient's actual visit and the database modification.
date_difference = """
SELECT 
    patient_id,
    clinical_event_date,
    db_system_write_date,
    (julianday(db_system_write_date) - julianday(clinical_event_date)) AS revision_delay_days
FROM clinical_snapshots
WHERE db_system_write_date > clinical_event_date;
"""
print("**************************** date_difference between clinical_event_date and db_system_write_date ************")
display(run_query(date_difference))

**************************** date_difference between clinical_event_date and db_system_write_date ************


,patient_id,clinical_event_date,db_system_write_date,revision_delay_days
0,P-62,2026-05-11,2026-06-02,22.0
1,P-65,2026-05-16,2026-06-04,19.0
